#AI Resume Screening System

#INSTALL DEPENDENCIES

In [ ]:
!pip install pdfplumber nltk scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 42.5 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 12.2.0
    Uninstalling pillow-12.2.0:
      Successfully uninstalled pillow-12.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.


#IMPORT LIBRARIES

In [ ]:
import pdfplumber
import nltk
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
import gradio as gr
import pdfplumber
import re
import nltk
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#DOWNLOAD NLP RESOURCES

In [ ]:
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


#UPLOAD RESUME PDF

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Subhan CV.PDF to Subhan CV.PDF


#EXTRACT TEXT FROM PDF

In [ ]:
resume_file = list(uploaded.keys())[0]

resume_text = ""

with pdfplumber.open(resume_file) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            resume_text += text

print(resume_text[:1000])

#SAMPLE JOB DESCRIPTION

In [ ]:
job_description = """
We are looking for an AI/ML Intern with skills in Python,
Machine Learning, Deep Learning, TensorFlow, SQL,
Data Analysis, Pandas, NumPy.
"""

#TEXT CLEANING FUNCTION

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

resume_clean = clean_text(resume_text)
jd_clean = clean_text(job_description)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

#CREATE TRAINING DATA

In [ ]:
data = {
    "text": [
        "python machine learning tensorflow deep learning data science",
        "sales marketing communication business management",
        "python pandas numpy sql data analysis machine learning",
        "hr recruitment communication leadership management",
        "deep learning neural networks tensorflow pytorch",
        "finance accounting excel budgeting analysis"
    ],
    "label": [1, 0, 1, 0, 1, 0]
}

import pandas as pd

df = pd.DataFrame(data)
df["clean_text"] = df["text"].apply(clean_text)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean_text"])
y = df["label"]

model = LogisticRegression()
model.fit(X, y)

LogisticRegression()

#FEATURE EXTRACTION (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["clean_text"])
y = df["label"]

#TRAIN ML MODEL

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

#MODEL EVALUATION

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



#PREDICT RESUME FIT

In [ ]:
resume_vec = vectorizer.transform([resume_clean])

prediction = model.predict(resume_vec)[0]

print("===== RESULT =====")

if prediction == 1:
    print("✅ Strong Technical Fit for AI/ML Role")
else:
    print("❌ Weak Fit for AI/ML Role")

===== RESULT =====
❌ Weak Fit for AI/ML Role


#BONUS SCORE

In [ ]:
prob = model.predict_proba(resume_vec)[0][1] * 100

print(f"Match Confidence Score: {prob:.2f}%")

Match Confidence Score: 49.56%


#GRADIO INTERFACE

In [ ]:
def analyze_resume(file, job_desc):

    # extract pdf text
    resume_text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                resume_text += page.extract_text()

    # clean
    resume_clean = clean_text(resume_text)

    # vectorize
    vec = vectorizer.transform([resume_clean])

    # prediction
    prob = model.predict_proba(vec)[0][1] * 100
    pred = model.predict(vec)[0]

    # output
    if pred == 1:
        result = "✅ Strong Match for AI/ML Role"
    else:
        result = "❌ Weak Match for AI/ML Role"

    return f"""
    📊 MATCH SCORE: {prob:.2f}%

    {result}

    📄 Resume Preview:
    {resume_text[:800]}
    """

#LAUNCH UI

In [ ]:
interface = gr.Interface(
    fn=analyze_resume,
    inputs=[
        gr.File(label="Upload Resume (PDF)"),
        gr.Textbox(label="Job Description")
    ],
    outputs=gr.Markdown(),
    title="AI Resume Screening System",
    description="Upload resume and check ML-based job fit"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3d56bfb8d2a1b2ed06.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
